# SFVP AI Clone — Full Pipeline
## Run top to bottom. No Google Drive needed.

**What it does:** You upload your voice + video → paste your script → get a finished video.

**Steps:**
1. Runtime → Change runtime type → **T4 GPU**
2. Runtime → **Run all**
3. When prompted, upload your files and paste your script
4. Wait ~15 min → download your finished video

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 0 — INSTALL (runs once, ~5-8 min)
# ═══════════════════════════════════════════════════
import os

print('Installing F5-TTS...')
!pip install git+https://github.com/SWivid/F5-TTS.git -q
!pip install soundfile -q

print('Installing MuseTalk...')
if not os.path.exists('/content/MuseTalk'):
    !git clone https://github.com/TMElyralab/MuseTalk.git /content/MuseTalk -q
!pip install -r /content/MuseTalk/requirements.txt -q
!pip install --upgrade diffusers transformers accelerate -q

print('Installing mmpose (required by MuseTalk)...')
# MuseTalk requirements break setuptools on Python 3.12 — fix it first
# Then use "python -m mim" (not shell "mim") so it picks up the upgraded packages
!pip install setuptools --upgrade -q
!pip install openmim --upgrade -q
!python -m mim install mmcv -q
!python -m mim install mmdet -q
!python -m mim install mmpose -q

print('Installing CodeFormer...')
if not os.path.exists('/content/CodeFormer'):
    !git clone https://github.com/sczhou/CodeFormer.git /content/CodeFormer -q
!pip install -r /content/CodeFormer/requirements.txt -q

!apt-get install -y ffmpeg -q
print('✓ All tools installed.')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 1 — DOWNLOAD MODEL WEIGHTS
#   Downloads ~5GB on first run. Redownloads each session
#   (Colab free tier doesn't persist between sessions).
# ═══════════════════════════════════════════════════
from huggingface_hub import snapshot_download

print('Downloading MuseTalk weights...')
snapshot_download('TMElyralab/MuseTalk',
                  local_dir='/content/MuseTalk/models/musetalk',
                  ignore_patterns=['*.md','*.txt'])

print('Downloading face landmark weights...')
snapshot_download('yzd-v/DWPose',
                  local_dir='/content/MuseTalk/models/dwpose',
                  ignore_patterns=['*.md'])

print('Downloading CodeFormer weights...')
%cd /content/CodeFormer
!python scripts/download_pretrained_models.py facelib -q
!python scripts/download_pretrained_models.py CodeFormer -q

print('All model weights ready.')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 2 — MOUNT DRIVE + BROWSE YOUR ASSETS
# ═══════════════════════════════════════════════════
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
except Exception:
    drive.mount('/content/drive', force_remount=True)

DRIVE_BASE = '/content/drive/MyDrive/SFVP_Clone'
os.makedirs(f'{DRIVE_BASE}/outputs', exist_ok=True)

voice_samples = [f for f in os.listdir(f'{DRIVE_BASE}/voice_samples')
                 if f.endswith(('.wav','.mp3','.m4a','.aac'))]
base_videos   = [f for f in os.listdir(f'{DRIVE_BASE}/base_videos')
                 if f.endswith(('.mp4','.mov'))] if os.path.exists(f'{DRIVE_BASE}/base_videos') else []
photos        = [f for f in os.listdir(f'{DRIVE_BASE}/photos')
                 if f.lower().endswith(('.jpg','.jpeg','.png'))] if os.path.exists(f'{DRIVE_BASE}/photos') else []

ft_model = f'{DRIVE_BASE}/fine_tuned_model/shennel_voice_checkpoint.pt'
has_fine_tuned = os.path.exists(ft_model)

print('════════════════════════════════')
print('  YOUR ASSETS')
print('════════════════════════════════')
print(f'Voice samples ({len(voice_samples)}):')
for i, s in enumerate(voice_samples): print(f'  [{i}] {s}')
print(f'\nBase videos ({len(base_videos)}):')
for i, v in enumerate(base_videos): print(f'  [{i}] {v}')
print(f'\nPhotos ({len(photos)}):')
for i, p in enumerate(photos): print(f'  [{i}] {p}')
print(f'\nFine-tuned voice model: {"✓ LOADED" if has_fine_tuned else "✗ Not yet (Notebook 05 creates this)"}')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 3 — ★ CONFIGURE YOUR VIDEO ★
#   Edit ONLY this cell before running.
# ═══════════════════════════════════════════════════

# ── YOUR SCRIPT ──────────────────────────────────────────────────────────────
SCRIPT = """
Sacramento, let me be real with you. If your brand doesn't stop people mid-scroll,
you're invisible. At Sactown's Finest, we do custom vinyl, banners, apparel,
and print — all in-house, right here in Sacramento. No middlemen.
No mystery markups. Just quality that speaks for itself.
We customize your life. DM us or hit the link in bio.
"""  # <-- REPLACE WITH YOUR SCRIPT EACH TIME

# ── VOICE SETTINGS ───────────────────────────────────────────────────────────
VOICE_REFERENCE = 'shennel_voice.wav'

USE_FINE_TUNED_MODEL = has_fine_tuned  # False until you run Notebook 05

# ── VIDEO SETTINGS ────────────────────────────────────────────────────────────
VIDEO_MODE = 'base_video'  # 'base_video' or 'photo'
VIDEO_SOURCE = 'rw-shennel-compressed.mp4'

# ★ SET THIS: how many seconds into the video until YOU appear on screen
# rw-shennel-compressed.mp4 → Shennel starts at ~49 seconds (skip the newscaster)
# Set to 0 if the video starts with you already on screen
VIDEO_START_SECONDS = 49

BBOX_SHIFT = 0

# ── EXPORT ───────────────────────────────────────────────────────────────────
FIDELITY = 0.7
EXPORT_FORMAT = 'reels'  # 'reels' (9:16) | 'square' (1:1) | 'wide' (16:9)
OUTPUT_FILENAME = 'FINAL_ad_v1.mp4'

# ─────────────────────────────────────────────────────────────────────────────
print('Configuration:')
print(f'  Script:    {len(SCRIPT.split())} words')
print(f'  Voice:     {VOICE_REFERENCE}  ← your own voice recording')
print(f'  Video:     {VIDEO_SOURCE}  (starting at {VIDEO_START_SECONDS}s)')
print(f'  Export:    {EXPORT_FORMAT} → {OUTPUT_FILENAME}')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 4 — GENERATE VOICE AUDIO
# ═══════════════════════════════════════════════════
from f5_tts.api import F5TTS
import subprocess, shutil

# Prepare reference audio
ref_src = f'{DRIVE_BASE}/voice_samples/{VOICE_REFERENCE}'
ref_wav = '/content/reference.wav'
subprocess.run(['ffmpeg','-y','-i',ref_src,'-t','10','-ar','24000','-ac','1',ref_wav],
               capture_output=True)

voice_output = '/content/voice_output.wav'

print('Loading F5-TTS model...')
if USE_FINE_TUNED_MODEL:
    tts = F5TTS(ckpt_file=ft_model)
    print('Using fine-tuned Shennel voice model.')
else:
    tts = F5TTS()
    print('Using zero-shot voice cloning.')

print('Generating voice audio...')
wav, sr, _ = tts.infer(
    ref_file=ref_wav,
    ref_text='',
    gen_text=SCRIPT.strip(),
    file_wave=voice_output,
    seed=42,
    remove_silence=True,
)

# Preview
from IPython.display import Audio, display
print('Voice preview:')
display(Audio(voice_output))
print('If this sounds wrong, stop here and try a different VOICE_REFERENCE in Step 3.')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 5 — GENERATE LIP-SYNCED VIDEO
# ═══════════════════════════════════════════════════

raw_video = '/content/raw_lipsync.mp4'

if VIDEO_MODE == 'base_video':
    src_video = f'{DRIVE_BASE}/base_videos/{VIDEO_SOURCE}'
    local_base = '/content/base_video_full.mp4'
    local_base_trimmed = '/content/base_video.mp4'

    print('Copying base video from Drive...')
    shutil.copy(src_video, local_base)

    # Get audio duration so we know how long to trim the base video
    dur_r = subprocess.run(
        ['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0', voice_output],
        capture_output=True, text=True)
    audio_dur = float(dur_r.stdout.strip()) + 2  # +2s buffer
    print(f'Audio duration: {audio_dur:.1f}s — cutting {VIDEO_START_SECONDS}s of intro, then {audio_dur:.1f}s of Shennel.')

    # Skip intro (newscaster etc.), grab only the segment where Shennel appears
    subprocess.run([
        'ffmpeg','-y',
        '-ss', str(VIDEO_START_SECONDS),   # seek to where Shennel starts
        '-i', local_base,
        '-t', str(audio_dur),              # grab only as much as we need
        '-c:v','libx264','-c:a','aac',
        local_base_trimmed
    ], capture_output=True)
    print('Base video trimmed to Shennel segment.')

    %cd /content/MuseTalk
    print('Running MuseTalk lip sync...')
    result = subprocess.run([
        'python', '-m', 'scripts.inference',
        '--video_path', local_base_trimmed,
        '--audio_path', voice_output,
        '--output_vid_name', raw_video,
        '--bbox_shift', str(BBOX_SHIFT),
        '--batch_size', '8',
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print('MuseTalk error — full log:')
        print(result.stdout[-3000:])
        print(result.stderr[-3000:])
        print('\nFalling back: merging trimmed video + audio (no lip sync).')
        subprocess.run([
            'ffmpeg','-y','-i', local_base_trimmed,'-i', voice_output,
            '-c:v','copy','-c:a','aac','-shortest', raw_video
        ], capture_output=True)
    else:
        print('✓ Lip sync complete.')

else:
    # ── PHOTO MODE (LivePortrait) ─────────────────────────────────────────────
    if not os.path.exists('/content/LivePortrait'):
        !git clone https://github.com/KwaiVGI/LivePortrait.git /content/LivePortrait -q
        !pip install -r /content/LivePortrait/requirements.txt -q
        from huggingface_hub import snapshot_download
        snapshot_download('KwaiVGI/LivePortrait',
                          local_dir='/content/LivePortrait/pretrained_weights',
                          ignore_patterns=['*.md','*.txt'])

    src_photo = f'{DRIVE_BASE}/photos/{VIDEO_SOURCE}'
    local_photo = '/content/source.jpg'
    shutil.copy(src_photo, local_photo)

    dur_r = subprocess.run(['ffprobe','-v','quiet','-show_entries','format=duration','-of','csv=p=0',voice_output],
                           capture_output=True,text=True)
    dur = float(dur_r.stdout.strip())

    driving = '/content/driving.mp4'
    if not os.path.exists(driving):
        !wget -q -O /content/driving.mp4 https://github.com/KwaiVGI/LivePortrait/raw/main/assets/examples/driving/d0.mp4

    driving_looped = '/content/driving_looped.mp4'
    subprocess.run(['ffmpeg','-y','-stream_loop','-1','-i',driving,
                    '-frames:v',str(int(dur*25)+10),'-c:v','libx264','-an',driving_looped],
                   capture_output=True)

    %cd /content/LivePortrait
    print('Running LivePortrait animation...')
    result = subprocess.run([
        'python','inference.py','-s',local_photo,'-d',driving_looped,
        '--output_dir','/content/lp_out','--flag_stitching'
    ], capture_output=True, text=True)

    import glob
    anim_vids = glob.glob('/content/lp_out/*.mp4')
    if anim_vids:
        subprocess.run(['ffmpeg','-y','-i',anim_vids[0],'-i',voice_output,
                        '-c:v','libx264','-c:a','aac','-shortest',raw_video],
                       capture_output=True)
        print('✓ Photo animation complete.')
    else:
        print('ERROR:', result.stderr[-1000:])

# Confirm raw video exists
if os.path.exists(raw_video):
    size = os.path.getsize(raw_video)
    print(f'raw_video ready: {size/1024/1024:.1f}MB')
else:
    print('ERROR: raw_video was not created. Check errors above.')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 6 — ENHANCE + EXPORT
# ═══════════════════════════════════════════════════
import glob

if not os.path.exists(raw_video):
    print('ERROR: No raw video found — check Step 5 output above.')
else:
    %cd /content/CodeFormer
    enhanced_dir = '/content/cf_enhanced'
    os.makedirs(enhanced_dir, exist_ok=True)

    print('Running CodeFormer face enhancement...')
    result = subprocess.run([
        'python', 'inference_codeformer.py',
        '-w', str(FIDELITY),
        '--input_path', raw_video,
        '--output_path', enhanced_dir,
        '--face_upsample',
        '--bg_upsampler', 'realesrgan',
        '--upscale', '2',
    ], capture_output=True, text=True)

    enhanced_vids = glob.glob(f'{enhanced_dir}/video/*.mp4')
    if not enhanced_vids:
        print('CodeFormer error — using raw video instead (still good quality).')
        print('Error detail:', result.stderr[-500:])
        enhanced_vid = raw_video
    else:
        enhanced_vid = enhanced_vids[0]
        print('Enhancement done.')

    # Export for social media
    FORMAT_MAP = {
        'reels':    'scale=1080:1920:force_original_aspect_ratio=increase,crop=1080:1920',
        'square':   'scale=1080:1080:force_original_aspect_ratio=increase,crop=1080:1080',
        'wide':     'scale=1920:1080:force_original_aspect_ratio=increase,crop=1920:1080',
        'original': 'scale=trunc(iw/2)*2:trunc(ih/2)*2',
    }
    scale_filter = FORMAT_MAP.get(EXPORT_FORMAT, FORMAT_MAP['original'])

    local_final = '/content/FINAL_output.mp4'
    export_result = subprocess.run([
        'ffmpeg','-y','-i', enhanced_vid,
        '-vf', scale_filter,
        '-c:v','libx264','-crf','18','-preset','fast',
        '-c:a','aac','-b:a','192k',
        '-movflags','+faststart',
        local_final
    ], capture_output=True, text=True)

    if not os.path.exists(local_final):
        print('FFmpeg export error:')
        print(export_result.stderr[-1000:])
    else:
        drive_final = f'{DRIVE_BASE}/outputs/{OUTPUT_FILENAME}'
        shutil.copy(local_final, drive_final)
        print(f'Saved to Drive: {drive_final}')
        print(f'File size: {os.path.getsize(local_final)/1024/1024:.1f}MB')

In [ ]:
# ═══════════════════════════════════════════════════
#   STEP 7 — PREVIEW + QUALITY CHECK
# ═══════════════════════════════════════════════════
from IPython.display import HTML
from base64 import b64encode

with open(local_final, 'rb') as f:
    vb64 = b64encode(f.read()).decode()

HTML(f'''
<h2 style="font-family:sans-serif">★ Your finished video ★</h2>
<video width="360" controls>
  <source src="data:video/mp4;base64,{vb64}" type="video/mp4">
</video>
<div style="font-family:sans-serif;margin-top:16px">
<b>Saved to Google Drive:</b> SFVP_Clone/outputs/{OUTPUT_FILENAME}<br><br>
<b>Not happy with it?</b><br>
• Voice sounds off → try a different VOICE_REFERENCE in Step 3<br>
• Lips look weird → adjust BBOX_SHIFT (try -5 to +5) in Step 3<br>
• Face looks over-smoothed → raise FIDELITY toward 0.9 in Step 3<br>
• Quality is generally low → Run Notebook 05 to fine-tune the voice model<br>
• Record more base videos in better lighting for sharper results<br><br>
<b>Ready to post?</b> Download from Google Drive → post directly to Instagram/TikTok.
</div>
''')